# Prep: aggregate benchmark results

Reads the raw results in `../../results` and aggregates them into one tidy long DataFrame, saved to `../data/benchmarks.parquet`.

Three sources, normalised into a common schema:

| source | harness | runtime | metric | unit |
|---|---|---|---|---|
| `cli-benchmark-results/*.json` (hyperfine) | `cli` | `nativeImage`, `python` | latency | s |
| `python-benchmark-results.txt` (JMH) | `jmh` | `python` | throughput | ops/s |
| `jvm-benchmark-result.txt` (JMH) | `jmh` | `jvm` | throughput | ops/s |

One row per (harness, runtime, schema, target).

In [32]:
import json
import re
import statistics
from pathlib import Path

import polars as pl

RESULTS = Path("../../results")
OUT = Path("../data/benchmarks.parquet")

# JMH method name -> canonical target name
TARGETS = {"jsonSchema": "json-schema", "shacl": "shacl"}

In [33]:
def parse_cli() -> pl.DataFrame:
    """hyperfine JSON: one file per (schema, target); commands are the runtimes.

    Latencies (s/run) are converted to throughput (runs/s) per individual run,
    so this harness reports throughput (ops/s) like JMH.
    """
    rows = []
    for f in sorted((RESULTS / "cli-benchmark-results").glob("*.json")):
        stem = f.stem  # e.g. 'chem-dcat-ap-json-schema'
        for target in ("json-schema", "shacl"):
            if stem.endswith("-" + target):
                schema = stem[: -len(target) - 1]
                break
        else:
            raise ValueError(f"cannot parse target from {f.name}")
        for r in json.loads(f.read_text())["results"]:
            thrpt = [1.0 / t for t in r["times"]]  # runs per second
            rows.append({
                "harness": "cli",
                "runtime": "scala" if r["command"] == "nativeImage" else "python",
                "schema": schema,
                "target": target,
                "mean": statistics.fmean(thrpt),
                "stddev": statistics.stdev(thrpt) if len(thrpt) > 1 else 0.0,
                "n": len(thrpt),
            })
    return pl.DataFrame(rows)

In [34]:
# Matches a JMH summary-table row in both formats:
#   LinkmlPython.jsonSchema  ai-atlas-nexus  thrpt  50   3.3447   0.1337  ops/s   (python: two columns)
#   WarmBench.jsonSchema     ai-atlas-nexus  thrpt  50  357.990 ±  3.303  ops/s   (jvm: ' ± ')
_JMH_ROW = re.compile(
    r"^\S+\.(?P<method>jsonSchema|shacl)\s+"
    r"(?P<model>\S+)\s+thrpt\s+(?P<cnt>\d+)\s+"
    r"(?P<score>[\d.]+)\s*(?:±\s*)?(?P<error>[\d.]+)\s+ops/s\s*$"
)


def parse_jmh(path: Path, runtime: str) -> pl.DataFrame:
    """Parse the JMH summary table from a benchmark log (throughput, ops/s)."""
    rows = []
    for line in path.read_text().splitlines():
        m = _JMH_ROW.match(line.strip())
        if not m:
            continue
        if m["model"] == "ai-atlas-nexus":
            continue # LinkML-Python SHACL gen errored out on this one
        rows.append({
            "harness": "jmh",
            "runtime": "scala" if runtime == "jvm" else "python",
            "schema": m["model"],
            "target": TARGETS[m["method"]],
            "mean": float(m["score"]),
            "stddev": float(m["error"]),
            "n": int(m["cnt"]),
        })
    if not rows:
        raise ValueError(f"no JMH rows parsed from {path}")
    return pl.DataFrame(rows)

In [35]:
df = pl.concat([
    parse_cli(),
    parse_jmh(RESULTS / "python-benchmark-results.txt", "python"),
    parse_jmh(RESULTS / "jvm-benchmark-result.txt", "jvm"),
], how="vertical")

df

harness,runtime,schema,target,mean,stddev,n
str,str,str,str,f64,f64,i64
"""cli""","""scala""","""brigde2ai_model_card""","""json-schema""",198.961643,8.411667,50
"""cli""","""python""","""brigde2ai_model_card""","""json-schema""",1.595661,0.024501,50
"""cli""","""scala""","""brigde2ai_model_card""","""shacl""",197.444068,8.27552,50
"""cli""","""python""","""brigde2ai_model_card""","""shacl""",1.55467,0.023737,50
"""cli""","""scala""","""cdm""","""json-schema""",2.819112,0.049155,50
…,…,…,…,…,…,…
"""jmh""","""scala""","""include""","""shacl""",2703.68,49.731,50
"""jmh""","""scala""","""iso27001""","""shacl""",671.576,7.008,50
"""jmh""","""scala""","""nmdc_microbiome""","""shacl""",3.765,0.056,50


In [36]:
# Sanity checks
print("shape:", df.shape)
print("schemas:", df["schema"].n_unique())
assert df["mean"].null_count() == 0
df.group_by("harness", "runtime", "target").len().sort("harness", "runtime", "target")

shape: (88, 7)
schemas: 11


harness,runtime,target,len
str,str,str,u32
"""cli""","""python""","""json-schema""",11
"""cli""","""python""","""shacl""",11
"""cli""","""scala""","""json-schema""",11
"""cli""","""scala""","""shacl""",11
"""jmh""","""python""","""json-schema""",11
"""jmh""","""python""","""shacl""",11
"""jmh""","""scala""","""json-schema""",11
"""jmh""","""scala""","""shacl""",11


In [37]:
df.group_by("schema").agg(pl.n_unique("harness"))

schema,harness
str,u32
"""iso27001""",2
"""crdch""",2
"""sssom""",2
"""d3fend""",2
"""chem-dcat-ap""",2
…,…
"""tc57cim""",2
"""cdm""",2
"""include""",2


In [38]:
OUT.parent.mkdir(parents=True, exist_ok=True)
df.write_parquet(OUT)
print(f"wrote {df.height} rows -> {OUT.resolve()}")

wrote 88 rows -> /home/piotr/neverblink/research/linkml-benchmark-schemas/analysis/data/benchmarks.parquet
